# Employee Case Analysis and Recommendation Model

This notebook analyzes `rcm.rcm.vw_bsml_daily_visit` at two levels:

1. **Historical analysis** by employee and case mix (`TestName` x `PayorName`)
2. **Daily analysis** using `production_date`, with extra focus on each employee's best days

It also builds a simple **recommendation model** to suggest which case combinations each employee performs best on.

## Output tables created in Python

- `df_raw`: raw data from the Snowflake view
- `daily_combo`: employee x production_date x testname x payorname
- `historical_combo`: historical summary by employee x testname x payorname
- `employee_best_day_overall`: each employee's best overall day
- `employee_best_day_combo`: each employee's best day for each test-payor combo
- `employee_recommendations`: ranked recommendation table for each employee

## Assumptions

- `vw_bsml_daily_visit` is already at the grain of:
  - `production_date`
  - `matched_employee_name`
  - `visitfid`
  - `TestName`
  - `PayorName`
- `total_cases_worked` in the view represents **touches/reworks** for that visit on that day.
- A **unique case** here is proxied by `visitfid`.


In [ ]:
# Install packages if needed
# Uncomment if your environment does not already have these.
# !pip install snowflake-connector-python pandas numpy matplotlib openpyxl sqlalchemy snowflake-sqlalchemy


In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from snowflake import connector

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)


## 1. Snowflake connection

Fill in your credentials below, or replace this block with your team's preferred auth method.


In [ ]:
CONFIG_FILE_PATH = os.path.abspath(r"C:\Users\25889\Desktop\BI\ANALYTICS\snowflake_config.json")

In [ ]:
def load_snowflake_config(config_path):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"Config file not found: {config_path}")

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    required_keys = [
        "account",
        "user",
        "authenticator",
        "role",
        "warehouse",
        "database",
        "schema"
    ]

    missing_keys = [key for key in required_keys if not config.get(key)]
    if missing_keys:
        raise ValueError(f"Missing required config keys: {missing_keys}")

    return config

In [ ]:
# SNOWFLAKE_CONFIG = {
#     'user': os.getenv('SNOWFLAKE_USER', 'YOUR_USER'),
#     'password': os.getenv('SNOWFLAKE_PASSWORD', 'YOUR_PASSWORD'),
#     'account': os.getenv('SNOWFLAKE_ACCOUNT', 'YOUR_ACCOUNT'),
#     'warehouse': os.getenv('SNOWFLAKE_WAREHOUSE', 'YOUR_WAREHOUSE'),
#     'database': os.getenv('SNOWFLAKE_DATABASE', 'RCM'),
#     'schema': os.getenv('SNOWFLAKE_SCHEMA', 'RCM'),
#     'role': os.getenv('SNOWFLAKE_ROLE', 'YOUR_ROLE')
# }

config = load_snowflake_config(CONFIG_FILE_PATH)

def get_snowflake_connection(config: dict):
    return connector.connect(
            account=config["account"],
            user=config["user"],
            authenticator=config["authenticator"],
            role=config["role"],
            warehouse=config["warehouse"],
            database=config["database"],
            schema=config["schema"]
        )


## 2. Load data from the view

You can optionally add a date filter in the SQL if you want to limit volume.


In [ ]:
SQL = '''
SELECT
    production_date,
    matched_employee_name,
    matched_email,
    matched_shore,
    visitfid,
    TestName,
    PayorName,
    total_cases_worked
FROM rcm.rcm.vw_bsml_daily_visit
WHERE matched_employee_name IS NOT NULL
  AND production_date IS NOT NULL
  AND visitfid IS NOT NULL
'''

with get_snowflake_connection(config) as conn:
    df_raw = pd.read_sql(SQL, conn)

print(df_raw.shape)
df_raw.head()


## 3. Data cleaning and standardization

In [ ]:
df = df_raw.copy()

# Standardize columns
rename_map = {
    'PRODUCTION_DATE': 'production_date',
    'MATCHED_EMPLOYEE_NAME': 'matched_employee_name',
    'MATCHED_EMAIL': 'matched_email',
    'MATCHED_SHORE': 'matched_shore',
    'VISITFID': 'visitfid',
    'TESTNAME': 'testname',
    'PAYORNAME': 'payorname',
    'TOTAL_CASES_WORKED': 'total_cases_worked'
}
df = df.rename(columns={c: rename_map.get(c, c.lower()) for c in df.columns})

df['production_date'] = pd.to_datetime(df['production_date']).dt.date

for col in ['matched_employee_name', 'matched_email', 'matched_shore', 'testname', 'payorname']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'': np.nan, 'None': np.nan, 'nan': np.nan})

# Fill missing categoricals with business-friendly labels
df['testname'] = df['testname'].fillna('UNKNOWN_TEST')
df['payorname'] = df['payorname'].fillna('UNKNOWN_PAYOR')
df['matched_shore'] = df['matched_shore'].fillna('UNKNOWN_SHORE')

df['total_cases_worked'] = pd.to_numeric(df['total_cases_worked'], errors='coerce').fillna(0)

# Each row in vw_bsml_daily_visit is one employee-day-visit combination.
# So this acts as a unique case row at the day level.
df['unique_case_count'] = 1

print(df.shape)
df.head()


## 4. Quick data quality checks

In [ ]:
summary = {
    'rows': len(df),
    'employees': df['matched_employee_name'].nunique(dropna=True),
    'tests': df['testname'].nunique(dropna=True),
    'payors': df['payorname'].nunique(dropna=True),
    'visits': df['visitfid'].nunique(dropna=True),
    'min_date': df['production_date'].min(),
    'max_date': df['production_date'].max(),
}
summary


In [ ]:
# Top missing / unknown rates
quality = pd.DataFrame({
    'column': ['testname', 'payorname', 'matched_shore'],
    'unknown_or_null_rows': [
        int((df['testname'] == 'UNKNOWN_TEST').sum()),
        int((df['payorname'] == 'UNKNOWN_PAYOR').sum()),
        int((df['matched_shore'] == 'UNKNOWN_SHORE').sum()),
    ],
    'pct_of_rows': [
        (df['testname'] == 'UNKNOWN_TEST').mean(),
        (df['payorname'] == 'UNKNOWN_PAYOR').mean(),
        (df['matched_shore'] == 'UNKNOWN_SHORE').mean(),
    ]
})
quality


## 5. Build the daily combo table

This is the most important dataset for your director's ask.

Grain:
- employee
- production_date
- testname
- payorname

Metrics:
- `daily_touches`: sum of touches/reworks
- `daily_unique_cases`: number of unique visit-level rows for that day and combo


In [ ]:
daily_combo = (
    df.groupby([
        'production_date',
        'matched_employee_name',
        'matched_email',
        'matched_shore',
        'testname',
        'payorname'
    ], dropna=False)
      .agg(
          daily_touches=('total_cases_worked', 'sum'),
          daily_unique_cases=('visitfid', 'nunique'),
          daily_case_rows=('unique_case_count', 'sum')
      )
      .reset_index()
)

# touches per unique case = how many times the same cases were worked/reworked on average
# lower may indicate cleaner processing; higher may indicate more rework/touches.
daily_combo['touches_per_case'] = (
    daily_combo['daily_touches'] / daily_combo['daily_unique_cases'].replace(0, np.nan)
)

print(daily_combo.shape)
daily_combo.head()


## 6. Historical analysis by employee x test x payor

This summarizes each employee's history for each test-payor combination.

Key columns:
- `active_days`: on how many days this combo appeared for the employee
- `historical_total_touches`
- `historical_total_unique_cases`
- `avg_daily_unique_cases`
- `max_daily_unique_cases`
- `consistency_cv_cases`: coefficient of variation; lower is more stable


In [ ]:
historical_combo = (
    daily_combo.groupby([
        'matched_employee_name',
        'matched_email',
        'matched_shore',
        'testname',
        'payorname'
    ], dropna=False)
    .agg(
        active_days=('production_date', 'nunique'),
        historical_total_touches=('daily_touches', 'sum'),
        historical_total_unique_cases=('daily_unique_cases', 'sum'),
        avg_daily_touches=('daily_touches', 'mean'),
        median_daily_touches=('daily_touches', 'median'),
        max_daily_touches=('daily_touches', 'max'),
        std_daily_touches=('daily_touches', 'std'),
        avg_daily_unique_cases=('daily_unique_cases', 'mean'),
        median_daily_unique_cases=('daily_unique_cases', 'median'),
        max_daily_unique_cases=('daily_unique_cases', 'max'),
        std_daily_unique_cases=('daily_unique_cases', 'std'),
        avg_touches_per_case=('touches_per_case', 'mean')
    )
    .reset_index()
)

historical_combo['std_daily_touches'] = historical_combo['std_daily_touches'].fillna(0)
historical_combo['std_daily_unique_cases'] = historical_combo['std_daily_unique_cases'].fillna(0)

historical_combo['consistency_cv_cases'] = (
    historical_combo['std_daily_unique_cases'] /
    historical_combo['avg_daily_unique_cases'].replace(0, np.nan)
)

historical_combo['consistency_cv_touches'] = (
    historical_combo['std_daily_touches'] /
    historical_combo['avg_daily_touches'].replace(0, np.nan)
)

historical_combo = historical_combo.sort_values(
    ['matched_employee_name', 'historical_total_unique_cases'],
    ascending=[True, False]
)

print(historical_combo.shape)
historical_combo.head(20)


## 7. Best historical combo for each employee

This answers:
- historically, which test-payor combo has the strongest volume for each employee?
- which combo gives them the highest average daily unique cases?


In [ ]:
# Best by total historical unique cases
employee_best_historical_volume = (
    historical_combo.sort_values(
        ['matched_employee_name', 'historical_total_unique_cases', 'avg_daily_unique_cases'],
        ascending=[True, False, False]
    )
    .groupby('matched_employee_name', as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# Best by average daily efficiency
employee_best_historical_efficiency = (
    historical_combo.sort_values(
        ['matched_employee_name', 'avg_daily_unique_cases', 'max_daily_unique_cases', 'active_days'],
        ascending=[True, False, False, False]
    )
    .groupby('matched_employee_name', as_index=False)
    .head(1)
    .reset_index(drop=True)
)

employee_best_historical_volume.head(20), employee_best_historical_efficiency.head(20)


## 8. Daily best-day analysis

This is the more important level.

We identify:
- each employee's best day overall
- each employee's best day for each test-payor combo


In [ ]:
# Overall best day per employee
employee_best_day_overall = (
    daily_combo.sort_values(
        ['matched_employee_name', 'daily_unique_cases', 'daily_touches'],
        ascending=[True, False, False]
    )
    .groupby('matched_employee_name', as_index=False)
    .head(1)
    .reset_index(drop=True)
)

employee_best_day_overall.head(10)


In [ ]:
# Best day per employee x test x payor
employee_best_day_combo = (
    daily_combo.sort_values(
        ['matched_employee_name', 'testname', 'payorname', 'daily_unique_cases', 'daily_touches'],
        ascending=[True, True, True, False, False]
    )
    .groupby(['matched_employee_name', 'testname', 'payorname'], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

employee_best_day_combo.head(10)


## 9. Daily pattern summary by employee x test x payor

This compresses the daily behavior while preserving daily context.


In [ ]:
daily_pattern_summary = (
    daily_combo.groupby([
        'matched_employee_name',
        'matched_email',
        'matched_shore',
        'testname',
        'payorname'
    ], dropna=False)
    .agg(
        observed_days=('production_date', 'nunique'),
        avg_daily_unique_cases=('daily_unique_cases', 'mean'),
        median_daily_unique_cases=('daily_unique_cases', 'median'),
        p90_daily_unique_cases=('daily_unique_cases', lambda s: np.percentile(s, 90)),
        max_daily_unique_cases=('daily_unique_cases', 'max'),
        avg_daily_touches=('daily_touches', 'mean'),
        p90_daily_touches=('daily_touches', lambda s: np.percentile(s, 90)),
        max_daily_touches=('daily_touches', 'max'),
        avg_touches_per_case=('touches_per_case', 'mean'),
        best_day=('production_date', 'max')
    )
    .reset_index()
)

daily_pattern_summary.head(10)


## 10. Recommendation model

This is a practical, explainable recommendation score instead of a black-box model.

### Why a score-based recommendation first?
This is easier to explain and trust.

### Score ingredients
- `avg_daily_unique_cases`: typical daily output on that combo
- `max_daily_unique_cases`: peak output on that combo
- `observed_days`: more evidence means more confidence
- `consistency`: lower variation is better

We can later replace this with an ML model if needed.


In [ ]:
rec = daily_combo.copy()

# Aggregate employee-combo daily behavior
rec = (
    rec.groupby(['matched_employee_name', 'matched_email', 'matched_shore', 'testname', 'payorname'], dropna=False)
       .agg(
           observed_days=('production_date', 'nunique'),
           avg_daily_unique_cases=('daily_unique_cases', 'mean'),
           max_daily_unique_cases=('daily_unique_cases', 'max'),
           avg_daily_touches=('daily_touches', 'mean'),
           max_daily_touches=('daily_touches', 'max'),
           std_daily_unique_cases=('daily_unique_cases', 'std'),
           std_daily_touches=('daily_touches', 'std')
       )
       .reset_index()
)

rec['std_daily_unique_cases'] = rec['std_daily_unique_cases'].fillna(0)
rec['std_daily_touches'] = rec['std_daily_touches'].fillna(0)

rec['consistency_score'] = 1 / (1 + rec['std_daily_unique_cases'])
rec['evidence_score'] = np.log1p(rec['observed_days'])

# Normalize metrics globally to keep scoring simple and transparent
def minmax(series: pd.Series) -> pd.Series:
    smin = series.min()
    smax = series.max()
    if pd.isna(smin) or pd.isna(smax) or smin == smax:
        return pd.Series(np.ones(len(series)), index=series.index)
    return (series - smin) / (smax - smin)

rec['avg_cases_norm'] = minmax(rec['avg_daily_unique_cases'])
rec['max_cases_norm'] = minmax(rec['max_daily_unique_cases'])
rec['evidence_norm'] = minmax(rec['evidence_score'])
rec['consistency_norm'] = minmax(rec['consistency_score'])

# Weighted recommendation score
rec['recommendation_score'] = (
    0.40 * rec['avg_cases_norm'] +
    0.30 * rec['max_cases_norm'] +
    0.20 * rec['evidence_norm'] +
    0.10 * rec['consistency_norm']
)

employee_recommendations = (
    rec.sort_values(
        ['matched_employee_name', 'recommendation_score', 'avg_daily_unique_cases', 'max_daily_unique_cases'],
        ascending=[True, False, False, False]
    )
    .copy()
)

employee_recommendations['employee_rank_for_combo'] = (
    employee_recommendations.groupby('matched_employee_name').cumcount() + 1
)

employee_recommendations.head(10)



## 11. Top recommended combinations for each employee

In [ ]:
top_n = 5

top_employee_recommendations = (
    employee_recommendations[employee_recommendations['employee_rank_for_combo'] <= top_n]
    .reset_index(drop=True)
)

top_employee_recommendations.head(10)


## 12. Example employee deep-dive

Set an employee name below to inspect:
- historical top combinations
- best days
- recent daily trend


In [ ]:
EMPLOYEE_NAME = 'ALINA RAI'  # Example: 'SAGARR' or exact matched_employee_name from the dataset

available_employees = sorted(df['matched_employee_name'].dropna().unique().tolist())
print(f'Employees available: {len(available_employees)}')
print(available_employees[:30])
print("yes" if EMPLOYEE_NAME in available_employees else "no")


In [ ]:
if EMPLOYEE_NAME:
    emp_daily = daily_combo[daily_combo['matched_employee_name'] == EMPLOYEE_NAME].copy()
    emp_hist = historical_combo[historical_combo['matched_employee_name'] == EMPLOYEE_NAME].copy()
    emp_best_combo = employee_best_day_combo[employee_best_day_combo['matched_employee_name'] == EMPLOYEE_NAME].copy()
    emp_recs = employee_recommendations[employee_recommendations['matched_employee_name'] == EMPLOYEE_NAME].copy()

    print('--- Historical top combos ---')
    display(emp_hist.sort_values(['avg_daily_unique_cases', 'historical_total_unique_cases'], ascending=False).head(5))

    print('--- Best day by combo ---')
    display(emp_best_combo.sort_values(['daily_unique_cases', 'daily_touches'], ascending=False).head(5))

    print('--- Recommendations ---')
    display(emp_recs.head(10))

    # Daily trend chart by total daily unique cases
    emp_daily_total = (
        emp_daily.groupby('production_date', as_index=False)
        .agg(total_daily_unique_cases=('daily_unique_cases', 'sum'))
        .sort_values('production_date')
    )

    plt.figure(figsize=(12, 5))
    plt.plot(emp_daily_total['production_date'], emp_daily_total['total_daily_unique_cases'])
    plt.title(f'Daily unique cases trend: {EMPLOYEE_NAME}')
    plt.xlabel('Production date')
    plt.ylabel('Unique cases')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Set EMPLOYEE_NAME to run the individual deep-dive section.')


## 13. Leaderboards

Useful for management review.


In [ ]:
# Top combos by historical average daily cases
leaderboard_avg_daily = historical_combo.sort_values(
    ['avg_daily_unique_cases', 'max_daily_unique_cases', 'active_days'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Top single best-day performances across all employees
leaderboard_best_days = daily_combo.sort_values(
    ['daily_unique_cases', 'daily_touches'],
    ascending=[False, False]
).reset_index(drop=True)

leaderboard_avg_daily.head(25), leaderboard_best_days.head(25)


## 14. Export outputs

This writes all important tables to CSV and one Excel workbook.


In [ ]:
# OUTPUT_DIR = 'employee_case_analysis_outputs'
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# export_frames = {
#     'daily_combo': daily_combo,
#     'historical_combo': historical_combo,
#     'employee_best_historical_volume': employee_best_historical_volume,
#     'employee_best_historical_efficiency': employee_best_historical_efficiency,
#     'employee_best_day_overall': employee_best_day_overall,
#     'employee_best_day_combo': employee_best_day_combo,
#     'daily_pattern_summary': daily_pattern_summary,
#     'employee_recommendations': employee_recommendations,
#     'top_employee_recommendations': top_employee_recommendations,
#     'leaderboard_avg_daily': leaderboard_avg_daily,
#     'leaderboard_best_days': leaderboard_best_days,
# }

# for name, frame in export_frames.items():
#     frame.to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), index=False)

# excel_path = os.path.join(OUTPUT_DIR, 'employee_case_analysis.xlsx')
# with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
#     for name, frame in export_frames.items():
#         frame.to_excel(writer, sheet_name=name[:31], index=False)

# print(f'Outputs written to: {OUTPUT_DIR}')
# print(f'Excel workbook: {excel_path}')


In [ ]:
import pandas as pd
import os

INPUT_DIR = 'employee_case_analysis_outputs'
output_path = os.path.join(INPUT_DIR, 'employee_case_analysis.xlsx')

csv_files = {
    # 'daily_combo': 'daily_combo.csv',
    'historical_combo': 'historical_combo.csv',
    'employee_best_historical_volume': 'employee_best_historical_volume.csv',
    'employee_best_historical_efficiency': 'employee_best_historical_efficiency.csv',
    'employee_best_day_overall': 'employee_best_day_overall.csv',
    'employee_best_day_combo': 'employee_best_day_combo.csv',
    'daily_pattern_summary': 'daily_pattern_summary.csv',
    'employee_recommendations': 'employee_recommendations.csv',
    'top_employee_recommendations': 'top_employee_recommendations.csv',
    'leaderboard_avg_daily': 'leaderboard_avg_daily.csv'
    # 'leaderboard_best_days': 'leaderboard_best_days.csv',
}

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    for sheet_name, file_name in csv_files.items():
        file_path = os.path.join(INPUT_DIR, file_name)
        df = pd.read_csv(file_path)
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"Excel file created at: {output_path}")

### ============================================================
## Recency-Weighted Recommendation Model
### ============================================================

In [ ]:
recency_df = daily_combo.copy()

# Make sure date is datetime
recency_df["production_date"] = pd.to_datetime(recency_df["production_date"])

# Latest date in dataset
latest_date = recency_df["production_date"].max()

# Days difference from latest date
recency_df["days_from_latest"] = (
    latest_date - recency_df["production_date"]
).dt.days

# Recency weight:
# recent days get higher weight, older days get lower weight
# half_life_days controls how fast old performance decays
half_life_days = 45

recency_df["recency_weight"] = 0.5 ** (
    recency_df["days_from_latest"] / half_life_days
)

# What it means:
# days_from_latest = 0 → weight = 1 (today)
# days_from_latest = 45 → weight = 0.5
# days_from_latest = 90 → weight = 0.25

# So performance gradually decays over time.

# Example
# Days Ago	Weight
# 0	        1.00
# 30	    ~0.63
# 60	    ~0.40
# 90	    ~0.25

# Interpretation: “Recent performance matters more, but older performance still contributes — just less.”

# Weighted metrics
recency_df["weighted_unique_cases"] = (
    recency_df["daily_unique_cases"] * recency_df["recency_weight"]
)

recency_df["weighted_touches"] = (
    recency_df["daily_touches"] * recency_df["recency_weight"]
)

# Aggregate at employee + TestName + PayorName level
recency_recommendations = (
    recency_df.groupby(
        [
            "matched_employee_name",
            "matched_email",
            "matched_shore",
            "testname",
            "payorname",
        ],
        dropna=False,
    )
    .agg(
        observed_days=("production_date", "nunique"),
        total_recency_weight=("recency_weight", "sum"),

        avg_daily_unique_cases=("daily_unique_cases", "mean"),
        max_daily_unique_cases=("daily_unique_cases", "max"),

        recency_weighted_avg_cases=(
            "weighted_unique_cases",
            "sum",
        ),

        avg_daily_touches=("daily_touches", "mean"),
        max_daily_touches=("daily_touches", "max"),

        std_daily_unique_cases=("daily_unique_cases", "std"),
    )
    .reset_index()
)

# Convert weighted sum into weighted average
recency_recommendations["recency_weighted_avg_cases"] = (
    recency_recommendations["recency_weighted_avg_cases"]
    / recency_recommendations["total_recency_weight"].replace(0, np.nan)
)

# Fill std nulls
recency_recommendations["std_daily_unique_cases"] = (
    recency_recommendations["std_daily_unique_cases"].fillna(0)
)

# Consistency score
recency_recommendations["consistency_score"] = (
    1 / (1 + recency_recommendations["std_daily_unique_cases"])
)

# Evidence score
recency_recommendations["evidence_score"] = np.log1p(
    recency_recommendations["observed_days"]
)

# Min-max normalization
def minmax(series: pd.Series) -> pd.Series:
    smin = series.min()
    smax = series.max()

    if pd.isna(smin) or pd.isna(smax) or smin == smax:
        return pd.Series(np.ones(len(series)), index=series.index)

    return (series - smin) / (smax - smin)


recency_recommendations["recency_avg_cases_norm"] = minmax(
    recency_recommendations["recency_weighted_avg_cases"]
)

recency_recommendations["avg_cases_norm"] = minmax(
    recency_recommendations["avg_daily_unique_cases"]
)

recency_recommendations["max_cases_norm"] = minmax(
    recency_recommendations["max_daily_unique_cases"]
)

recency_recommendations["evidence_norm"] = minmax(
    recency_recommendations["evidence_score"]
)

recency_recommendations["consistency_norm"] = minmax(
    recency_recommendations["consistency_score"]
)

# New recommendation score with recency included
recency_recommendations["recency_recommendation_score"] = (
    0.35 * recency_recommendations["recency_avg_cases_norm"] +
    0.25 * recency_recommendations["avg_cases_norm"] +
    0.20 * recency_recommendations["max_cases_norm"] +
    0.10 * recency_recommendations["evidence_norm"] +
    0.10 * recency_recommendations["consistency_norm"]
)

# Rank recommendations per employee
recency_recommendations = (
    recency_recommendations.sort_values(
        [
            "matched_employee_name",
            "recency_recommendation_score",
            "recency_weighted_avg_cases",
            "avg_daily_unique_cases",
            "max_daily_unique_cases",
        ],
        ascending=[True, False, False, False, False],
    )
    .reset_index(drop=True)
)

recency_recommendations["employee_recency_rank_for_combo"] = (
    recency_recommendations.groupby("matched_employee_name").cumcount() + 1
)

# Top 5 recommendations per employee
top_recency_recommendations = (
    recency_recommendations[
        recency_recommendations["employee_recency_rank_for_combo"] <= 5
    ]
    .reset_index(drop=True)
)

recency_recommendations.head()

In [ ]:
OUTPUT_DIR = 'employee_case_analysis_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

export_frames = {
    'recency_recommendations': recency_recommendations,
    # 'historical_combo': historical_combo,
}

for name, frame in export_frames.items():
    frame.to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), index=False)